<a href="https://colab.research.google.com/github/Srideep-Kundu/MLProjects/blob/main/DynamicTolling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import json

with open('/content/traffic.json', 'r') as json_data :
  data = json.load(json_data)

df = pd.DataFrame(data['records']).head(10000)
df.head()

,trip_id,driver_id,route,state,distance_km,duration_minutes,avg_speed_kmh,behavior_class,behavior_label,sensor_features,safety_score,grade,weather,time_of_day,traffic_density,accident_risk,timestamp
0,T000001,D23133,NH-1 Delhi-Amritsar,Gujarat,184.8,260,42.7,0,normal_driving,"[0.158, 0, 0, 0.372, -0.085, 0.456, 0.036, 0.1...",36,F,fog,night,high,0.15,1730821520
1,T000002,D29720,NH-8 Delhi-Mumbai,Uttar Pradesh,130.3,169,46.1,0,normal_driving,"[0.078, 0, 0, 0.228, -0.07, 0.298, 0.036, 0.07...",39,F,clear,peak_evening,high,0.17,1742426943
2,T000003,D08650,NH-3 Agra-Mumbai,Delhi,125.7,156,48.5,0,normal_driving,"[0.127, 0, 0, 0.374, -0.086, 0.459, 0.045, 0.1...",30,F,fog,night,high,0.14,1738703905
3,T000004,D31719,Yamuna Expressway,Rajasthan,165.0,212,46.6,2,sharp_turn,"[0.153, 0, 0, 0.4, -0.096, 0.496, 4.209, 13.80...",37,F,heavy_rain,midday,high,0.06,1745921559
4,T000005,D26385,Yamuna Expressway,Maharashtra,129.0,128,60.5,3,aggressive_acceleration,"[2.796, 0, 0, 2.959, 2.64, 0.319, 0.176, 0.028...",50,F,cloudy,night,high,0.06,1751267102


# Task
Select 7 features (include traffic density / traffic volume, time of day etc) and feed them to a price-prediction model (regression or reinforcement learning) to recommend fair tolls. Split the data into training set, cross-validation set and test set and then scale the data. Also address issues of underfitting and overfitting.

## Data preparation

### Subtask:
Select the relevant features, handle categorical variables (if any), and split the data into training, cross-validation, and test sets.


**Reasoning**:
Select the specified features, create dummy variables for categorical features, define the target variable, and split the data into training, cross-validation, and test sets.



In [2]:
from sklearn.model_selection import train_test_split

# Select features
features = ['traffic_density', 'time_of_day', 'distance_km', 'duration_minutes', 'avg_speed_kmh', 'weather', 'accident_risk']
X = df[features]
y = df['safety_score']

# Create dummy variables for categorical features
X = pd.get_dummies(X, columns=['traffic_density', 'time_of_day', 'weather'], drop_first=True)

# Split data into training (60%), validation (20%), and test (20%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Training set shape: (6000, 14) (6000,)
Validation set shape: (2000, 14) (2000,)
Test set shape: (2000, 14) (2000,)


## Feature scaling

### Subtask:
Scale the numerical features to ensure they have a similar range.


**Reasoning**:
Scale the numerical features in the training, validation, and test sets using StandardScaler.



In [3]:
from sklearn.preprocessing import StandardScaler

# Identify numerical columns (exclude the one-hot encoded columns)
numerical_cols = ['distance_km', 'duration_minutes', 'avg_speed_kmh', 'accident_risk']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the numerical columns of the training data
scaler.fit(X_train[numerical_cols])

# Transform the numerical columns in all three sets
X_train[numerical_cols] = scaler.transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

display(X_train.head())

,distance_km,duration_minutes,avg_speed_kmh,accident_risk,traffic_density_low,traffic_density_medium,traffic_density_very_high,time_of_day_night,time_of_day_peak_evening,time_of_day_peak_morning,weather_cloudy,weather_fog,weather_heavy_rain,weather_light_rain
9761,-1.272444,-1.030780,-0.432238,0.053974,False,True,False,False,False,False,False,False,False,False
3159,-1.380770,-1.377163,1.005308,0.630207,False,False,False,False,False,False,False,False,False,False
1489,-0.226481,-0.484562,0.593328,1.494557,False,False,False,False,True,False,True,False,False,False
2987,-1.508629,-1.323873,-0.011493,0.053974,False,True,False,False,True,False,False,True,False,False
812,-0.849797,-0.697721,-0.300756,-1.098492,False,False,True,False,True,False,True,False,False,False


## Model selection

### Subtask:
Choose a suitable regression model for price prediction.


**Reasoning**:
Choose a suitable regression model, considering the problem and data characteristics. Linear models are a good starting point for regression tasks.



In [4]:
from sklearn.linear_model import LinearRegression

# Linear Regression is a simple and interpretable model, suitable as a baseline.
# It assumes a linear relationship between features and the target variable.
# Given the nature of the features (some numerical, some one-hot encoded categorical)
# and the target (safety score), a linear model can capture potential linear dependencies.
# More complex models can be considered later if needed.
model = LinearRegression()

## Model training

### Subtask:
Train the selected regression model on the training data.


**Reasoning**:
Train the Linear Regression model using the scaled training data.



In [5]:
model.fit(X_train, y_train)

LinearRegression()

## Model evaluation

### Subtask:
Evaluate the model's performance on the cross-validation set and address issues of underfitting and overfitting.


**Reasoning**:
Evaluate the model's performance on the cross-validation set using MSE and R-squared, and print the results to assess potential underfitting or overfitting.



In [6]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions on the cross-validation set
y_pred_val = model.predict(X_val)

# Calculate Mean Squared Error (MSE)
mse_val = mean_squared_error(y_val, y_pred_val)

# Calculate R-squared score
r2_val = r2_score(y_val, y_pred_val)

# Print the performance metrics
print(f"Mean Squared Error on Cross-Validation Set: {mse_val:.2f}")
print(f"R-squared Score on Cross-Validation Set: {r2_val:.2f}")

# Note: To fully assess underfitting/overfitting, training set performance would also be needed for comparison.
# However, based on the validation set performance alone, we can get an initial indication.
# A low R-squared and high MSE might suggest underfitting or that the linear model is not suitable.
# If training metrics were much better than validation metrics, it would suggest overfitting.

Mean Squared Error on Cross-Validation Set: 47.47
R-squared Score on Cross-Validation Set: 0.81


## Final evaluation

### Subtask:
Evaluate the final model on the test set.


**Reasoning**:
Evaluate the trained model on the test set by making predictions, calculating MSE and R-squared, and printing the results.



In [7]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions on the test set
y_pred_test = model.predict(X_test)

# Calculate Mean Squared Error (MSE) on the test set
mse_test = mean_squared_error(y_test, y_pred_test)

# Calculate R-squared score on the test set
r2_test = r2_score(y_test, y_pred_test)

# Print the performance metrics for the test set
print(f"Mean Squared Error on Test Set: {mse_test:.2f}")
print(f"R-squared Score on Test Set: {r2_test:.2f}")

# Compare test set performance with cross-validation performance
print(f"\nComparison with Cross-Validation Performance:")
print(f"Cross-Validation MSE: {mse_val:.2f}")
print(f"Test Set MSE: {mse_test:.2f}")
print(f"Cross-Validation R-squared: {r2_val:.2f}")
print(f"Test Set R-squared: {r2_test:.2f}")

# Briefly confirm generalization ability
if abs(mse_test - mse_val) < 5 and abs(r2_test - r2_val) < 0.05:
    print("\nTest set performance is similar to cross-validation performance, indicating good generalization.")
elif r2_test >= r2_val and mse_test <= mse_val:
     print("\nTest set performance is slightly better or similar to cross-validation performance, indicating good generalization.")
else:
    print("\nThere is some difference between test set and cross-validation performance. Further analysis might be needed.")

Mean Squared Error on Test Set: 47.22
R-squared Score on Test Set: 0.82

Comparison with Cross-Validation Performance:
Cross-Validation MSE: 47.47
Test Set MSE: 47.22
Cross-Validation R-squared: 0.81
Test Set R-squared: 0.82

Test set performance is similar to cross-validation performance, indicating good generalization.


## Toll recommendation

### Subtask:
Use the trained model to recommend fair tolls based on the selected features.


**Reasoning**:
Define a function to recommend fair tolls based on predicted safety score and other factors, then demonstrate it with an example.



In [9]:
# Define the toll recommendation function
def recommend_toll(features, model, scaler, actual_safety_score=None):
    """
    Recommends a fair toll based on vehicle and traffic features.

    Args:
        features (dict): A dictionary of features for a single trip.
                         Keys should match the original feature names.
        model: The trained regression model (e.g., LinearRegression).
        scaler: The fitted StandardScaler used for numerical features.
        actual_safety_score (float, optional): The actual safety score for the trip.
                                               Defaults to None.

    Returns:
        tuple: A tuple containing the recommended toll amount, the predicted safety score,
               and the actual safety score (if provided).
    """
    # Convert features dictionary to a pandas DataFrame
    features_df = pd.DataFrame([features])

    # Create dummy variables for categorical features, ensuring columns match training data
    categorical_cols = ['traffic_density', 'time_of_day', 'weather']
    features_df = pd.get_dummies(features_df, columns=categorical_cols, drop_first=True)

    # Align columns with the training data - crucial for consistent prediction
    # Add missing columns with default value 0
    for col in X_train.columns:
        if col not in features_df.columns:
            features_df[col] = 0
    # Ensure the order of columns is the same as in the training data
    features_df = features_df[X_train.columns]

    # Identify numerical columns and scale them using the fitted scaler
    numerical_cols = ['distance_km', 'duration_minutes', 'avg_speed_kmh', 'accident_risk']
    features_df[numerical_cols] = scaler.transform(features_df[numerical_cols])

    # Predict the safety score using the trained model
    predicted_safety_score = model.predict(features_df)[0]

    # Define the toll recommendation logic (example logic)
    # This is a simplified logic; a real-world scenario would be more complex.
    # Higher safety score generally means lower risk, so a lower toll might be justified.
    # We can also factor in traffic density - higher density might warrant a higher toll.

    # Example logic: Basic toll + adjustment based on safety score and traffic density
    base_toll = 50.0 # Example base toll
    safety_score_adjustment = (100 - predicted_safety_score) * 0.5 # Lower safety score adds to toll
    # Assume 'traffic_density_high' column exists after one-hot encoding
    traffic_density_adjustment = 0
    if 'traffic_density_high' in features_df.columns and features_df['traffic_density_high'].iloc[0] == 1:
        traffic_density_adjustment = 20 # Add to toll for high traffic density
    elif 'traffic_density_medium' in features_df.columns and features_df['traffic_density_medium'].iloc[0] == 1:
        traffic_density_adjustment = 10 # Add to toll for medium traffic density


    recommended_toll = base_toll + safety_score_adjustment + traffic_density_adjustment

    # Ensure the recommended toll is not negative
    recommended_toll = max(10.0, recommended_toll) # Minimum toll

    return recommended_toll, predicted_safety_score, actual_safety_score

# Demonstrate the function with an example
example_features = {
    'traffic_density': 'high',
    'time_of_day': 'peak_evening',
    'distance_km': 150.0,
    'duration_minutes': 180,
    'avg_speed_kmh': 50.0,
    'weather': 'clear',
    'accident_risk': 0.1
}

# Example with an actual safety score for comparison
example_actual_safety_score = 70  # Replace with an actual safety score if available

recommended_toll_amount, predicted_safety, actual_safety = recommend_toll(example_features, model, scaler, actual_safety_score=example_actual_safety_score)

print(f"Based on the provided features:")
print(f"  Recommended toll: ₹{recommended_toll_amount:.2f}")
print(f"  Predicted safety score: {predicted_safety:.2f}")
if actual_safety is not None:
    print(f"  Actual safety score: {actual_safety:.2f}")

Based on the provided features:
  Recommended toll: ₹62.44
  Predicted safety score: 75.12
  Actual safety score: 70.00


## Summary:

### Data Analysis Key Findings

* The dataset was successfully split into training (60%), validation (20%), and test (20%) sets, resulting in 6000 training samples and 2000 samples each for validation and testing.
* Categorical features ('traffic\_density', 'time\_of\_day', 'weather') were one-hot encoded, increasing the number of features from 7 to 14.
* Numerical features ('distance\_km', 'duration\_minutes', 'avg\_speed\_kmh', 'accident\_risk') were successfully scaled using `StandardScaler`.
* A Linear Regression model was chosen for the price prediction task.
* The Linear Regression model achieved an R-squared score of 0.81 and a Mean Squared Error (MSE) of 47.47 on the cross-validation set.
* The model's performance on the test set was very similar to the validation set, with an R-squared of 0.82 and an MSE of 47.22. This similarity indicates good generalization ability.

### Insights or Next Steps

* The current model demonstrates a reasonably good fit, explaining approximately 81-82% of the variance in the target variable.
* Explore more complex models (e.g., polynomial regression, tree-based models) to see if they can capture non-linear relationships and potentially improve performance, while carefully monitoring for overfitting.
* The `recommend_toll` function provides a basic example of how the predicted safety score and other features can be used to suggest a toll. This logic can be further refined based on business requirements and additional factors.
* Consider using a reinforcement learning approach to dynamically adjust tolls based on real-time traffic conditions and other factors to optimize traffic flow and revenue.